# Relation Extraction Pipeline

Extracts **entity→entity edges** from cached chunks. All edges are typed `RELATES_TO`
and carry a plain-English `relation_description` — one sentence from the source explaining
how the two entities are connected. No standardized vocabulary, no closed enum.

**Why `RELATES_TO` + description instead of typed predicates:**
- PPR only needs an edge to walk — it does not need a predicate label to traverse the graph.
- Natural-language descriptions preserve the actual semantics better than a closed vocab
  that forces every relation into `hasPart` / `usesTool` / `other`.
- Downstream: you can cluster descriptions or embed them to recover semantic groupings
  without having baked in a vocabulary decision at extraction time.

**Pipeline (3 stages):**
1. **Stage 1** — LLM finds entity pairs and writes one sentence per pair.
2. **Stage 2** — Object resolution: object name → canonical entity ID via `mention_to_canonical`.
3. **Stage 3** — Span verification + confidence scoring.

**Inputs:** `entity_cache.json`, `resolved_entities.json`  
**Output:** `relations_cache.json` — `RelationEdge` records staged for Neo4j write.

## Setup & imports

In [68]:
import json
import re
import sys
import hashlib
from pathlib import Path
from typing import NamedTuple

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()
client = OpenAI()

ENTITY_CACHE    = Path("entity_cache.json")
RESOLVED_CACHE  = Path("resolved_entities.json")
RELATIONS_CACHE = Path("relations_cache.json")

EXTRACTOR_VERSION = "relation-extraction-v1/gpt-4.1-mini"
EXTRACT_MODEL     = "gpt-4.1-mini"

for p in (ENTITY_CACHE, RESOLVED_CACHE):
    if not p.exists():
        sys.exit(f"missing cache: {p}")
print("caches found ✓")

caches found ✓


## RelationEdge model

`relation_description` is the atomic fact — a single sentence copied or lightly paraphrased
from the source that explains the connection. Both entity endpoints are stored; the object
may be unresolved if the object entity was not extracted upstream.

In [71]:
FACT_TYPE_VALUES = [
    "assertion",       # checkable entity-to-entity claim — the ONLY type stored as an edge
    "definition",
    "opinion",
    "instruction",
    "metaphor",
    "prediction",
    "other_non_fact",
]


class RelationEdge(BaseModel):
    """Entity-to-entity RELATES_TO edge with a plain-English description.

    All edges in the graph are :RELATES_TO. The semantic content lives in
    `relation_description`, not in a predicate label.
    """

    relation_id:          str            # hash of (doc_id, char_start, subject, object_raw)

    # endpoints
    subject_entity_id:    str            # canonical, from the entity list
    relation_description: str            # one sentence: how subject and object are connected
    object_raw:           str            # verbatim object name from source
    object_entity_id:     str | None = None   # canonical, resolved via mention_to_canonical
    object_resolved:      bool        = False

    # fact classification
    fact_type:            str         = "assertion"

    # bi-temporal — opt-in; default = current state
    valid_from:           str | None  = None
    valid_to:             str | None  = None
    temporal_inferred:    bool        = False

    # provenance
    doc_id:               str
    char_start:           int
    char_end:             int
    source_quote:         str

    # pipeline stamps
    span_verified:        bool         = False
    confidence:           float        = 1.0
    extractor_version:    str          = EXTRACTOR_VERSION


class ChunkResult(NamedTuple):
    edges:              list[RelationEdge]
    non_facts:          list[dict]   # rejected by fact-type gate
    rejected_subjects:  list[dict]   # subject not in canonical entity list
    unresolved_objects: list[dict]   # object not in mention_to_canonical


def make_relation_id(
    doc_id: str, char_start: int, subject: str, object_raw: str
) -> str:
    key = f"{doc_id}|{char_start}|{subject}|{object_raw}"
    return hashlib.sha1(key.encode()).hexdigest()[:16]


print("RelationEdge model loaded ✓")

RelationEdge model loaded ✓


## Load caches

In [74]:
resolved_data         = json.loads(RESOLVED_CACHE.read_text())
mention_to_canonical: dict[str, str] = resolved_data["mention_to_canonical"]
print(f"resolved entities : {resolved_data['total_entities']:>5}")
print(f"surface forms     : {len(mention_to_canonical):>5}")

entity_cache = json.loads(ENTITY_CACHE.read_text())
mentions     = entity_cache["mentions"]

_doc_cache: dict[str, str] = {}

def _doc_text(source_file: str) -> str:
    if source_file not in _doc_cache:
        p = Path(source_file)
        _doc_cache[source_file] = p.read_text(encoding="utf-8", errors="ignore") if p.exists() else ""
    return _doc_cache[source_file]


ChunkKey = tuple[str, str]
chunks: dict[ChunkKey, dict] = {}

for m in mentions:
    key: ChunkKey = (m["source_file"], str(m["episode_idx"]))
    if key in chunks:
        continue
    episode = m["episode"]
    doc     = _doc_text(m["source_file"])
    offset  = doc.find(episode)
    chunks[key] = {
        "doc_id":      m["source_file"],
        "episode_idx": str(m["episode_idx"]),
        "text":        episode,
        "char_start":  offset,
        "char_end":    offset + len(episode) if offset >= 0 else -1,
    }

chunk_entities: dict[ChunkKey, list[str]] = {k: [] for k in chunks}
for m in mentions:
    key = (m["source_file"], str(m["episode_idx"]))
    canon = mention_to_canonical.get(m["text"])
    if canon and canon not in chunk_entities[key]:
        chunk_entities[key].append(canon)

bad = sum(1 for c in chunks.values() if c["char_start"] < 0)
print(f"\nchunks            : {len(chunks):>5}")
print(f"bad char offsets  : {bad:>5}")

resolved entities :  1026
surface forms     :   880

chunks            :   182
bad char offsets  :    33


## Stage 1 — LLM extraction

The model finds entity pairs and writes one sentence per pair. No vocabulary.
The sentence should be tight enough to be an atomic fact — one claim, not a summary.

In [77]:
_SYSTEM_PROMPT = """\
You read a text chunk and identify pairs of named entities that are connected.
For each pair, write ONE sentence describing how they are related.

Rules:
- The sentence must be an atomic fact — one claim, grounded in the text.
- Copy or lightly paraphrase the source; do not infer beyond what is stated.
- Skip scalar attributes: headcounts, dates, money amounts, role titles.
  'the team has 12 people' is NOT a relation — it's a scalar fact, skip it.
- Skip non-facts: opinions, metaphors, instructions, definitions, predictions.

EXAMPLES of good relation_description sentences:
  'GraphRAG uses Community detection to cluster entity nodes into communities.'
  'Alice works for Acme Corp as a data scientist.'
  'SharePoint is part of Microsoft 365 and hosts team documents.'
  'The Startup Owners Manual was written by Steve Blank.'
  'Azure Monitor uses Application Insights to collect telemetry.'

EXAMPLES of what to skip:
  'Opinions are worthless'        -> opinion, skip
  'the team has 12 people'        -> scalar headcount, skip
  'X is like a warehouse'         -> metaphor, skip
  'never talk about the idea'     -> instruction, skip

You will receive:
- TEXT: the source chunk
- ENTITIES: canonical entity IDs — the ONLY valid subjects

For each relation, emit:
  subject_entity_id    — copy VERBATIM from the ENTITIES list
  raw_object_str       — the object entity name VERBATIM from the text
  relation_description — one sentence stating how subject and object are connected
  fact_type            — assertion | definition | opinion | instruction | metaphor | prediction | other_non_fact
  valid_from           — ISO date only if explicitly stated in the text; else null
  valid_to             — ISO date only if explicitly stated; else null
  temporal_inferred    — true if you inferred a time period rather than reading it
"""

_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "relations": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "subject_entity_id":    {"type": "string"},
                    "raw_object_str":       {"type": "string"},
                    "relation_description": {"type": "string"},
                    "fact_type":            {"type": "string", "enum": FACT_TYPE_VALUES},
                    "valid_from":           {"type": ["string", "null"]},
                    "valid_to":             {"type": ["string", "null"]},
                    "temporal_inferred":    {"type": "boolean"},
                },
                "required": [
                    "subject_entity_id", "raw_object_str", "relation_description",
                    "fact_type", "valid_from", "valid_to", "temporal_inferred",
                ],
            },
        }
    },
    "required": ["relations"],
}


def extract_relations_from_chunk(chunk: dict, entities: list[str]) -> list[dict]:
    """Stage 1: LLM finds entity pairs and writes one sentence per pair."""
    if not entities:
        return []
    user_msg = (
        f"TEXT:\n{chunk['text']}\n\n"
        "ENTITIES:\n" + "\n".join(f"- {e}" for e in entities)
    )
    resp = client.chat.completions.create(
        model=EXTRACT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": _SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "relations",
                "strict": True,
                "schema": _SCHEMA,
            },
        },
    )
    return json.loads(resp.choices[0].message.content)["relations"]


print("Stage 1 ready ✓")
print("  output: (subject, object, one-sentence description) — no predicate vocabulary")

Stage 1 ready ✓
  output: (subject, object, one-sentence description) — no predicate vocabulary


## Stage 2 — Object resolution

Resolves `raw_object_str` to a canonical entity ID. Case-insensitive fallback.
Unresolved = the object entity was not extracted upstream; logged but edge is kept.

In [40]:
def resolve_object(raw_object_str: str) -> tuple[str | None, bool]:
    """Stage 2: raw object name -> (canonical_id, resolved)."""
    canon = mention_to_canonical.get(raw_object_str)
    if canon:
        return canon, True
    raw_lower = raw_object_str.lower()
    for surface, cid in mention_to_canonical.items():
        if surface.lower() == raw_lower:
            return cid, True
    return None, False


print("Stage 2 object resolver ready ✓")

Stage 2 object resolver ready ✓


## Stage 3 — Span verification + confidence

Checks that `raw_object_str` appears verbatim in `source_quote`.

**Confidence formula:**
```
start                    1.00
object_unresolved       -0.20   object not in mention_to_canonical
span_verified=False     -0.30   object name not found verbatim in source
temporal_inferred       -0.10   valid-time inferred, not stated
```

In [43]:
_DIGITS_RE = re.compile(r"\d+")


def verify_span(
    raw_object_str: str,
    source_quote: str,
    valid_from: str | None,
    valid_to: str | None,
) -> dict:
    """Stage 3: check raw_object_str appears in source_quote."""
    span_verified = bool(raw_object_str) and raw_object_str.lower() in source_quote.lower()

    temporal_inferred = False
    for date_val in filter(None, [valid_from, valid_to]):
        digits = _DIGITS_RE.findall(date_val)
        if digits and not all(d in source_quote for d in digits):
            temporal_inferred = True
            break

    return {"span_verified": span_verified, "temporal_inferred": temporal_inferred}


def score_confidence(
    object_resolved: bool,
    span_verified: bool,
    temporal_inferred: bool,
) -> float:
    """Stage 3 confidence for RELATES_TO edges."""
    score = 1.00
    if not object_resolved:  score -= 0.20
    if not span_verified:    score -= 0.30
    if temporal_inferred:    score -= 0.10
    return round(max(0.0, score), 2)


print("Stage 3 verify + confidence ready ✓")
print("  formula: 1.0 - 0.20(unresolved) - 0.30(span_fail) - 0.10(temporal)")

Stage 3 verify + confidence ready ✓
  formula: 1.0 - 0.20(unresolved) - 0.30(span_fail) - 0.10(temporal)


## Full pipeline

In [46]:
def process_chunk(chunk: dict, entities: list[str]) -> ChunkResult:
    """Run all stages on one chunk and return RELATES_TO RelationEdges."""
    raw_items = extract_relations_from_chunk(chunk, entities)

    edges:              list[RelationEdge] = []
    non_facts:          list[dict]         = []
    rejected_subjects:  list[dict]         = []
    unresolved_objects: list[dict]         = []

    entity_set = set(entities)

    for rr in raw_items:
        fact_type = rr.get("fact_type", "assertion")
        raw_obj   = rr["raw_object_str"]
        desc      = rr["relation_description"]

        # Gate: non-assertions go to the non_fact log
        if fact_type != "assertion":
            non_facts.append({
                "fact_type":           fact_type,
                "subject_entity_id":   rr["subject_entity_id"],
                "raw_object_str":      raw_obj,
                "relation_description":desc,
                "doc_id":              chunk["doc_id"],
            })
            continue

        # Subject must be in the canonical entity list
        subj = rr["subject_entity_id"]
        if subj not in entity_set:
            rejected_subjects.append({
                "subject_attempted":   subj,
                "raw_object_str":      raw_obj,
                "relation_description":desc,
                "doc_id":              chunk["doc_id"],
            })
            continue

        # Stage 2: object resolution
        obj_canon, obj_resolved = resolve_object(raw_obj)
        if not obj_resolved:
            unresolved_objects.append({
                "subject_entity_id":   subj,
                "raw_object_str":      raw_obj,
                "relation_description":desc,
                "doc_id":              chunk["doc_id"],
                "note":                "object not in mention_to_canonical",
            })

        # Stage 3: span verify + confidence
        verify = verify_span(raw_obj, chunk["text"],
                             rr.get("valid_from"), rr.get("valid_to"))
        temporal_inf = verify["temporal_inferred"] or rr.get("temporal_inferred", False)
        conf = score_confidence(
            object_resolved   = obj_resolved,
            span_verified     = verify["span_verified"],
            temporal_inferred = temporal_inf,
        )

        edges.append(RelationEdge(
            relation_id          = make_relation_id(chunk["doc_id"], chunk["char_start"], subj, raw_obj),
            subject_entity_id    = subj,
            relation_description = desc,
            object_raw           = raw_obj,
            object_entity_id     = obj_canon,
            object_resolved      = obj_resolved,
            fact_type            = fact_type,
            valid_from           = rr.get("valid_from"),
            valid_to             = rr.get("valid_to"),
            temporal_inferred    = temporal_inf,
            doc_id               = chunk["doc_id"],
            char_start           = chunk["char_start"],
            char_end             = chunk["char_end"],
            source_quote         = chunk["text"],
            span_verified        = verify["span_verified"],
            confidence           = conf,
        ))

    return ChunkResult(
        edges=edges,
        non_facts=non_facts,
        rejected_subjects=rejected_subjects,
        unresolved_objects=unresolved_objects,
    )


print("pipeline assembled ✓")

pipeline assembled ✓


## Demo run — 10 chunks

In [50]:
from collections import Counter

DEMO_CHUNKS = 5

demo_edges:      list[RelationEdge] = []
demo_non_facts:  list[dict]         = []
demo_rejected:   list[dict]         = []
demo_unresolved: list[dict]         = []

chunk_items = [
    (key, chunk) for key, chunk in chunks.items()
    if chunk["char_start"] >= 0 and chunk_entities.get(key)
][:DEMO_CHUNKS]

for key, chunk in chunk_items:
    try:
        result = process_chunk(chunk, chunk_entities[key])
        demo_edges.extend(result.edges)
        demo_non_facts.extend(result.non_facts)
        demo_rejected.extend(result.rejected_subjects)
        demo_unresolved.extend(result.unresolved_objects)
    except Exception as e:
        print(f"  chunk {key} failed: {e}")

total = len(demo_edges) + len(demo_non_facts) + len(demo_rejected)
resolved_n = sum(1 for e in demo_edges if e.object_resolved)

print("=" * 64)
print(f"  RELATION EDGES  ({DEMO_CHUNKS} chunks)")
print("=" * 64)
print(f"  total LLM claims    : {total}")
print(f"  edges kept          : {len(demo_edges)}")
print(f"  non-facts rejected  : {len(demo_non_facts)}")
for ft, cnt in sorted(Counter(n['fact_type'] for n in demo_non_facts).items(), key=lambda x: -x[1]):
    print(f"    {ft:<22} {cnt}")
print(f"  invalid subjects    : {len(demo_rejected)}")
print()
print(f"  object resolved     : {resolved_n}/{len(demo_edges)} "
      f"({100*resolved_n//max(1, len(demo_edges))}%)")
print(f"  unresolved (logged) : {len(demo_unresolved)}")
print()

if demo_edges:
    confs = sorted(e.confidence for e in demo_edges)
    buckets = Counter(round(c, 1) for c in confs)
    print(f"  CONFIDENCE  min={confs[0]:.2f}  max={confs[-1]:.2f}  mean={sum(confs)/len(confs):.2f}")
    for score in sorted(buckets):
        print(f"    {score:.1f}  {'█' * buckets[score]} ({buckets[score]})")
    print()

print("=" * 64)
print(f"  ALL EDGES ({len(demo_edges)})")
print("=" * 64)
for e in sorted(demo_edges, key=lambda x: -x.confidence):
    res_mark = "✓" if e.object_resolved else "?"
    sv_mark  = "✓" if e.span_verified   else "✗"
    print(f"  [{e.confidence:.2f}] obj={res_mark} span={sv_mark}")
    print(f"    {e.subject_entity_id}")
    print(f"    -RELATES_TO-> {e.object_raw!r}" + (f"  [{e.object_entity_id}]" if e.object_entity_id else ""))
    print(f"    '{e.relation_description}'")
    print(f"    {Path(e.doc_id).name}  [{e.char_start}:{e.char_end}]")
    print()

  RELATION EDGES  (5 chunks)
  total LLM claims    : 26
  edges kept          : 25
  non-facts rejected  : 1
    opinion                1
  invalid subjects    : 0

  object resolved     : 25/25 (100%)
  unresolved (logged) : 0

  CONFIDENCE  min=1.00  max=1.00  mean=1.00
    1.0  █████████████████████████ (25)

  ALL EDGES (25)
  [1.00] obj=✓ span=✓
    Compound Annual Growth Rate
    -RELATES_TO-> 'revenue'  [revenue]
    'Compound Annual Growth Rate measures how fast revenue grows per year over a multi-year period.'
    CAGR.md  [0:208]

  [1.00] obj=✓ span=✓
    Compound Annual Growth Rate
    -RELATES_TO-> 'market'  [market]
    'Compound Annual Growth Rate measures how fast a market grows per year over a multi-year period.'
    CAGR.md  [0:208]

  [1.00] obj=✓ span=✓
    Compound Annual Growth Rate
    -RELATES_TO-> 'year'  [year]
    'Compound Annual Growth Rate assumes growth compounds each year, meaning each year's growth builds on the last.'
    CAGR.md  [0:208]

  [1.00] obj

## Bulk run — all chunks → `relations_cache.json`

In [52]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm as tqdm_nb

BULK_WORKERS = 8
SAVE_EVERY   = 50

eligible = [
    (key, chunk) for key, chunk in chunks.items()
    if chunk["char_start"] >= 0 and chunk_entities.get(key)
]
print(f"{len(eligible)} eligible chunks")

all_edges:      list[dict] = []
all_non_facts:  list[dict] = []
all_unresolved: list[dict] = []


def _process(args: tuple) -> tuple[list[dict], list[dict], list[dict]]:
    key, chunk = args
    result = process_chunk(chunk, chunk_entities[key])
    return (
        [e.model_dump(mode="json") for e in result.edges],
        result.non_facts,
        result.unresolved_objects,
    )


def _save() -> None:
    RELATIONS_CACHE.write_text(json.dumps(all_edges, ensure_ascii=False, indent=2))


with ThreadPoolExecutor(max_workers=BULK_WORKERS) as executor:
    futures = {executor.submit(_process, item): item for item in eligible}
    with tqdm_nb(total=len(eligible), desc="relation edges") as bar:
        for i, future in enumerate(as_completed(futures)):
            eb, nfb, urb = future.result()
            all_edges.extend(eb)
            all_non_facts.extend(nfb)
            all_unresolved.extend(urb)
            bar.update(1)
            if (i + 1) % SAVE_EVERY == 0:
                _save()

_save()

resolved_n = sum(1 for e in all_edges if e.get("object_resolved"))
conf_vals  = [e["confidence"] for e in all_edges]

print(f"\n{len(all_edges)} edges  |  {len(all_non_facts)} non-facts  |  {len(all_unresolved)} unresolved objects")
print(f"  object resolved : {resolved_n}/{len(all_edges)} ({100*resolved_n//max(1, len(all_edges))}%)")
print(f"  avg confidence  : {sum(conf_vals)/max(1, len(conf_vals)):.2f}")
print(f"  saved to        : {RELATIONS_CACHE}")

98 eligible chunks


relation edges:   0%|          | 0/98 [00:00<?, ?it/s]


481 edges  |  7 non-facts  |  172 unresolved objects
  object resolved : 309/481 (64%)
  avg confidence  : 0.91
  saved to        : relations_cache.json


## Graph write — next step

Every edge writes as `:RELATES_TO`. The `relation_description` property holds the
plain-English sentence and is what you display, search, or embed for similarity.

```cypher
MATCH (s:Entity {canonical_id: $subject_entity_id})
MATCH (o:Entity {canonical_id: $object_entity_id})
MERGE (s)-[r:RELATES_TO {relation_id: $relation_id}]->(o)
SET r.relation_description = $relation_description,
    r.source_quote         = $source_quote,
    r.doc_id               = $doc_id,
    r.confidence           = $confidence,
    r.valid_from           = $valid_from,
    r.valid_to             = $valid_to
```

Skip edges where `object_entity_id` is null — no target node to connect to.

**PPR note:** PPR walks `:RELATES_TO` edges regardless of description.
To filter by semantic type later, embed `relation_description` vectors and
cluster at query time rather than locking in a label at extraction time.